In [3]:
import pandas as pd

package_data = pd.read_csv('../../package_data.csv')
first_release_date = pd.read_csv('first_releases.csv')

In [7]:
## need to extract the unique system / package pairs 
system_data = package_data[['package_name', 'System']].drop_duplicates()
system_data.to_csv('package_and_system.csv', index=False)

release_data = first_release_date[['package', 'platform']].drop_duplicates()

## I added repositories 1/1/26 and need to see which repositories need a first release date, compared to what I collected previously

In [11]:
## which packages are in system data but not in release data?
missing_packages = system_data[~system_data['package_name'].isin(release_data['package'])]
missing_packages.to_csv('package_and_system_rerun.csv', index=False)


In [ ]:
import requests
import json
import csv
import time
from itertools import cycle
from urllib.parse import quote

# Add your three API keys here
API_KEYS = [
    "e59a8912721943821d1a64641c632d3e",
    "46343d7b6a359c4c2be819604934b4c4",
    "32c074d39acc781896027b14c2812686"
]

# Create a rotating key iterator
key_rotation = cycle(API_KEYS)

def get_first_release(platform, package_name, api_key):
    """Get first release date for a package - same logic as original"""
    # URL-encode the package name to handle slashes and special characters
    encoded_package = quote(package_name, safe='')
    url = f"https://libraries.io/api/{platform}/{encoded_package}"
    
    response = requests.get(url, params={'api_key': api_key})
    
    if response.status_code == 200:
        data = response.json()
        
        # Check if versions are embedded
        if 'versions' in data:
            versions = data['versions']
            print(f"Found {len(versions)} versions for {package_name}")
            
            # Get first release
            dates = []
            for v in versions:
                if isinstance(v, dict) and 'published_at' in v:
                    dates.append(v['published_at'])
            
            if dates:
                first_date = min(dates)
                print(f"First release: {first_date.split('T')[0]}")
                return first_date.split('T')[0]
        else:
            print(f"No 'versions' key in response for {package_name}")
            print("\nAll keys in response:")
            for key in data.keys():
                value = data[key]
                if isinstance(value, list):
                    print(f"  {key}: [list with {len(value)} items]")
                elif isinstance(value, dict):
                    print(f"  {key}: [dict]")
                else:
                    print(f"  {key}: {value}")
    else:
        print(f"Error {response.status_code} for {package_name}")
        print(f"URL attempted: {url}")
    
    return None

def save_results_to_csv(results, output_file, mode='w'):
    """Save results to CSV file"""
    with open(output_file, mode, newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['package', 'platform', 'first_release'])
        if mode == 'w':
            writer.writeheader()
        writer.writerows(results)

# Read packages from CSV
input_file = 'package_and_system_rerun.csv'  # Changed to your CSV filename
output_file = 'first_releases_run2.csv'
results = []

with open(input_file, 'r') as f:
    reader = csv.DictReader(f)
    
    for i, row in enumerate(reader, 1):
        # Changed to read 'package_name' and 'System' columns
        package_name = row['package_name']
        platform = row['System'].lower()  # Convert NPM/PYPI to npm/pypi
        
        # Get next API key in rotation
        current_key = next(key_rotation)
        
        print(f"\n[Package {i}] Processing {platform}/{package_name}")
        print(f"Using API key #{API_KEYS.index(current_key) + 1}")
        print("-" * 50)
        
        first_release = get_first_release(platform, package_name, current_key)
        
        results.append({
            'package': package_name,
            'platform': platform,
            'first_release': first_release or 'Not found'
        })
        
        # Save results every 25 packages
        if i % 25 == 0:
            print(f"\n*** Saving checkpoint at {i} packages ***")
            save_results_to_csv(results, output_file)
            print(f"*** Checkpoint saved to {output_file} ***\n")
        
        # With 3 keys, you can make 180 requests per minute (60 per key)
        # Sleep less since load is distributed across keys
        time.sleep(0.35)  # ~3 requests per second across all keys

# Final save of all results
save_results_to_csv(results, output_file)

print(f"\n\nResults saved to {output_file}")
print(f"Processed {len(results)} packages")
print(f"Used {len(API_KEYS)} API keys in rotation")

## Which packages were not found via the API and can we find these manually? 

In [13]:
date_df= pd.read_csv('first_releases.csv')

## print the packages with first_release as 'Not found'
not_found_packages = date_df[date_df['first_release'] == 'Not found']
print(not_found_packages)


                                 package platform first_release
14173      nf-workflow-ui-5>2.10.0>glogg      npm     Not found
14627                      clubhouse-lib      npm     Not found
14663  @aensley/semantic-release-openapi      npm     Not found


In [16]:
## Need to manually add the first release dates for these packages
date_df.loc[date_df['package'] == 'nuxt-simple-robots', 'first_release'] = '2022-12-19' #https://libraries.io/npm/nuxt-simple-robots
date_df.loc[date_df['package'] == 'react-native-video-player', 'first_release'] = '2016-09-07' #https://libraries.io/npm/react-native-video-player
date_df.loc[date_df['package'] == 'momento-wire-types', 'first_release'] = '2022-02-09' #https://libraries.io/pypi/momento-wire-types
date_df.loc[date_df['package'] == 'django-oso', 'first_release'] = '2020-09-08' #https://libraries.io/pypi/django-oso
date_df.loc[date_df['package'] == '@aensley/semantic-release-openapi', 'first_release'] = '2022-10-12' #https://libraries.io/npm/@aensley%2Fsemantic-release-openapi
date_df.loc[date_df['package'] == 'clubhouse-lib', 'first_release'] = '2016-05-20' #https://libraries.io/npm/clubhouse-lib
date_df.loc[date_df['package'] == 'nf-workflow-ui-5>2.10.0>glogg', 'first_release'] = '2019-06-19' #https://libraries.io/npm/nf-workflow-ui-5

In [17]:
## verify the step above worked 
not_found_packages = date_df[date_df['first_release'] == 'Not found']
print(not_found_packages)


Empty DataFrame
Columns: [package, platform, first_release]
Index: []


In [18]:
## save the results to a csv file 
date_df.to_csv('first_releases.csv', index=False)